# Conditioning-ladder checkpoint diagnosis on Modal

This read-only experiment diagnoses the three saved transformers from training run `20260826T001237667541Z`. It does not train or save model weights.

For every checkpoint it measures dense local flow behavior at `t = 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00`, compares correct and swapped NVARC contexts for the seen panel at the same flow state, continues target-path rollouts from `t = 0.50, 0.90, 0.99, 1.00`, and repeats normal 50-step generation with two fixed seeds. Noise is paired across conditions and checkpoints. The local and partial probes deliberately use the encoded target path; only normal generation is target-free and consults Ground Truth after prediction.

Success means a durable `COMPLETE` report plus clearly labelled grid and full-30x30 comparison sheets for all three checkpoints. The known deterministic VAE error is retained as an explicit round-trip reference rather than attributed to the transformer.


In [ ]:
!nvidia-smi


In [ ]:
%uv pip uninstall torchvision


## 1. Immutable configuration

Set `DIFFUSION_ARC_REVISION` to the exact pushed 40-character commit that contains this notebook and its diagnosis runner. The historical training commit and run ID below are intentionally fixed.


In [ ]:
from __future__ import annotations

import base64
from datetime import datetime, timezone
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import subprocess
import sys
import time

VOLUME_NAME = 'diffusion-arc-models'
VOLUME_ROOT = Path('/mnt/diffusion-arc-models')
REPOSITORY_URL = 'https://github.com/joshuakauling/diffusion-arc.git'
# Set this environment variable before running, or replace the right-hand side with the pushed commit.
SOURCE_REVISION = os.environ.get('DIFFUSION_ARC_REVISION', '').strip()
TRAINING_SOURCE_REVISION = '202a10316bc48d18e7d6e98755de87d118b5a032'
TRAINING_RUN_ID = '20260826T001237667541Z'
DIAGNOSIS_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
LOCAL_OUTPUT_ROOT = Path('/tmp/diffusion-arc-conditioning-diagnosis')
LOCAL_REPORT_PATH = LOCAL_OUTPUT_ROOT / DIAGNOSIS_ID / 'report.json'
AUTO_SHUTDOWN = os.environ.get('DIFFUSION_ARC_AUTO_SHUTDOWN', '1') == '1'
EXPECTED_STAGES = (
    'one-condition',
    'two-condition-routing',
    'eight-condition-overfit',
)

if len(SOURCE_REVISION) != 40 or any(
    character not in '0123456789abcdefABCDEF' for character in SOURCE_REVISION
):
    raise ValueError(
        'DIFFUSION_ARC_REVISION must be the exact pushed 40-character commit'
    )
if not VOLUME_ROOT.is_dir():
    raise FileNotFoundError(f'Modal Volume mount is missing: {VOLUME_ROOT}')
if LOCAL_REPORT_PATH.exists():
    raise FileExistsError(LOCAL_REPORT_PATH)
print('Diagnosis ID:', DIAGNOSIS_ID)
print('Diagnosis source:', SOURCE_REVISION)
print('Parent training run:', TRAINING_RUN_ID)
print('Local report:', LOCAL_REPORT_PATH)


## 2. Fetch, install, and test the exact source

The checkout is detached at the selected commit. Package pins are installed only when they differ, and the complete unit-test suite must pass before any checkpoint is loaded.


In [ ]:
def run_command(label: str, command: list[str], *, cwd: Path | None = None, env=None) -> str:
    completed = subprocess.run(
        command, cwd=cwd, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, check=False,
    )
    if completed.returncode != 0:
        tail = '\n'.join(completed.stdout.splitlines()[-50:])
        if tail:
            print(tail)
        raise RuntimeError(f'{label} failed with exit code {completed.returncode}')
    return completed.stdout.strip()

SOURCE_DIR = Path('/tmp') / f'diffusion-arc-diagnosis-source-{DIAGNOSIS_ID}'
SOURCE_DIR.mkdir(parents=True)
run_command('git init', ['git', 'init'], cwd=SOURCE_DIR)
run_command('add origin', ['git', 'remote', 'add', 'origin', REPOSITORY_URL], cwd=SOURCE_DIR)

github_token = os.environ.get('GITHUB_TOKEN', '').strip()
if not github_token:
    raise RuntimeError('Attach a Modal Secret containing GITHUB_TOKEN')
credential = base64.b64encode(
    f'x-access-token:{github_token}'.encode('utf-8')
).decode('ascii')
git_environment = os.environ.copy()
git_environment.pop('GITHUB_TOKEN', None)
git_environment.update({
    'GIT_TERMINAL_PROMPT': '0',
    'GIT_CONFIG_COUNT': '1',
    'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
    'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {credential}',
})
try:
    run_command(
        'fetch source',
        ['git', 'fetch', '--depth=1', 'origin', SOURCE_REVISION],
        cwd=SOURCE_DIR, env=git_environment,
    )
    run_command('checkout source', ['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=SOURCE_DIR)
finally:
    git_environment.clear()
    del credential
RESOLVED_SOURCE_COMMIT = run_command(
    'resolve source', ['git', 'rev-parse', 'HEAD'], cwd=SOURCE_DIR
)
if RESOLVED_SOURCE_COMMIT.lower() != SOURCE_REVISION.lower():
    raise RuntimeError('Fetched source commit differs from requested revision')

PINNED_PACKAGES = {
    'accelerate': '1.12.0', 'diffusers': '0.39.0', 'einops': '0.8.1',
    'Pillow': '11.3.0', 'safetensors': '0.8.0', 'torch': '2.13.0',
    'tqdm': '4.67.1', 'transformers': '4.56.1',
}
def installed_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
requirements = [
    f'{name}=={version}' for name, version in PINNED_PACKAGES.items()
    if installed_version(name) != version
]
if requirements:
    run_command(
        'install pinned dependencies',
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input',
         '--no-cache-dir', *requirements],
    )
run_command(
    'install Diffusion-ARC source',
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input',
     '--no-deps', '--no-build-isolation', '-e', str(SOURCE_DIR)],
)
source_path = str((SOURCE_DIR / 'src').resolve())
if source_path not in sys.path:
    sys.path.insert(0, source_path)
importlib.invalidate_caches()
test_environment = os.environ.copy()
test_environment['PYTHONDONTWRITEBYTECODE'] = '1'
test_environment['PYTHONPATH'] = source_path
test_started = time.perf_counter()
test_process = subprocess.run(
    [sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests'],
    cwd=SOURCE_DIR, env=test_environment, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, check=False,
)
print('\n'.join(test_process.stdout.splitlines()[-30:]))
if test_process.returncode != 0:
    raise RuntimeError(f'Unit tests returned {test_process.returncode}')
print('Resolved source commit:', RESOLVED_SOURCE_COMMIT)
print(f'Unit tests: PASS in {time.perf_counter() - test_started:.1f}s')


## 3. Run the diagnosis

The runner validates every checkpoint file against the hashes in the historical report, caches all 17 NVARC contexts and VAE targets once, then evaluates the three transformers sequentially. Progress is printed for every checkpoint, timestep, partial rollout, and generation seed. Verified report and image snapshots are uploaded after every phase, and the known harmless Modal `asyncio` callback-noise block is filtered from the stream.


In [ ]:
runner = SOURCE_DIR / 'scripts' / 'experiments' / 'diagnose_conditioning_ladder_checkpoints.py'
if not runner.is_file():
    raise RuntimeError(
        'Selected source revision lacks the diagnosis runner; push the implementation '
        'and select that exact commit.'
    )
runner_command = [
    sys.executable, '-u', str(runner),
    '--volume-root', str(VOLUME_ROOT),
    '--volume-name', VOLUME_NAME,
    '--training-run-id', TRAINING_RUN_ID,
    '--diagnosis-id', DIAGNOSIS_ID,
    '--source-commit', RESOLVED_SOURCE_COMMIT,
    '--expected-training-source-commit', TRAINING_SOURCE_REVISION,
    '--local-output-root', str(LOCAL_OUTPUT_ROOT),
]
runner_environment = os.environ.copy()
runner_environment['PYTHONPATH'] = source_path
process = subprocess.Popen(
    runner_command, cwd=SOURCE_DIR, env=runner_environment,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
assert process.stdout is not None
noise_lines_to_skip = 0
for output_line in process.stdout:
    if output_line.rstrip() == 'Exception in callback None()':
        noise_lines_to_skip = 5
        continue
    if noise_lines_to_skip:
        noise_lines_to_skip -= 1
        continue
    print(output_line, end='')
return_code = process.wait()
if return_code != 0:
    if LOCAL_REPORT_PATH.is_file():
        failed_report = json.loads(LOCAL_REPORT_PATH.read_text(encoding='utf-8'))
        print('Failed stage:', failed_report.get('failed_stage'))
        print('Failures:', failed_report.get('failures'))
    raise RuntimeError(f'Diagnosis runner exited with {return_code}')
REPORT = json.loads(LOCAL_REPORT_PATH.read_text(encoding='utf-8'))
if REPORT.get('status') != 'COMPLETE':
    raise RuntimeError(f'Diagnosis report is not COMPLETE: {REPORT.get("status")}')
durability = REPORT.get('durability', {})
if not durability.get('remote_file_sizes_verified'):
    raise RuntimeError('Remote artifact sizes were not verified')
if not durability.get('remote_report_content_verified'):
    raise RuntimeError('Remote report content was not verified')
REPORT['summary']


## 4. Compact result and artifact index


In [ ]:
artifact_index = {}
for stage_name in EXPECTED_STAGES:
    stage = REPORT['checkpoints'][stage_name]
    if stage.get('status') != 'COMPLETE':
        raise RuntimeError(f'{stage_name} diagnosis is not COMPLETE')
    artifact_index[stage_name] = {
        'full_generation': [
            item['artifacts'] for item in stage['full_generation']['by_seed']
        ],
        'partial_rollout_sheets': [
            assignment['artifacts']
            for entry in stage['partial_rollouts']['entries']
            for assignment in entry['assignments']
        ],
    }
{
    'diagnosis_id': DIAGNOSIS_ID,
    'local_report': str(LOCAL_REPORT_PATH),
    'remote_report': REPORT['durability']['remote_report_path'],
    'summary': REPORT['summary'],
    'artifact_index': artifact_index,
}


## 5. Preview the most relevant sheets

These previews show the eight-condition checkpoint at normal generation, the direct local `t=1` reconstruction, and the `t=1 → 0` rollout. Every row includes Ground Truth, VAE round-trip, raw prediction, classified prediction, and a red mismatch map. All remaining sheets are stored beside the report.


In [ ]:
from IPython.display import Image as DisplayImage, display

stage = REPORT['checkpoints']['eight-condition-overfit']
preview_paths = [
    stage['full_generation']['by_seed'][0]['artifacts']['grid_comparison'],
    stage['local_flow_probe']['groups']['seen']['entries'][-1]
        ['assignments'][0]['artifacts']['grid_comparison'],
    stage['partial_rollouts']['entries'][-1]
        ['assignments'][0]['artifacts']['grid_comparison'],
]
for relative_path in preview_paths:
    path = LOCAL_REPORT_PATH.parent / relative_path
    print(relative_path)
    display(DisplayImage(filename=str(path)))


## 6. Optional automatic shutdown

Shutdown is scheduled only after all three diagnoses are complete, every remote artifact size matches, and the final report has been read back byte-for-byte. Set `DIFFUSION_ARC_AUTO_SHUTDOWN=0` before starting to keep the kernel alive.


In [ ]:
durable_report = json.loads(LOCAL_REPORT_PATH.read_text(encoding='utf-8'))
if durable_report.get('status') != 'COMPLETE':
    raise RuntimeError('Refusing shutdown: report is not COMPLETE')
durability = durable_report.get('durability', {})
if not durability.get('remote_file_sizes_verified'):
    raise RuntimeError('Refusing shutdown: remote artifact sizes are unverified')
if not durability.get('remote_report_content_verified'):
    raise RuntimeError('Refusing shutdown: remote report content is unverified')
for stage_name in EXPECTED_STAGES:
    if durable_report['checkpoints'][stage_name].get('status') != 'COMPLETE':
        raise RuntimeError(f'Refusing shutdown: {stage_name} is incomplete')
if not AUTO_SHUTDOWN:
    print('Automatic shutdown disabled; all diagnosis artifacts are durable.')
else:
    modal_task_id = os.environ.get('MODAL_TASK_ID', '').strip()
    if not modal_task_id:
        raise RuntimeError('MODAL_TASK_ID is unavailable; stop the kernel manually')
    shutdown_code = (
        'import subprocess, sys, time; time.sleep(10); '
        "subprocess.run(['modal', 'container', 'stop', '--yes', sys.argv[1]], check=False)"
    )
    subprocess.Popen(
        [sys.executable, '-c', shutdown_code, modal_task_id],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    print('All artifacts verified. Modal shutdown scheduled in 10 seconds.')


## Interpretation guide

- A good local reconstruction at low `t` but a sharp collapse near `t=1` shows that the high-noise region was not learned sufficiently under the current training allocation; it does not mean that this region was never sampled.
- At `t=1`, every row in a local seen panel has exactly the same noisy image latent. Differences between correct and swapped contexts therefore isolate conditioning behavior.
- A correct-context partial rollout that succeeds from `t=0.5` but fails from `t=0.99` or `1.0` shows where the learned vector field stops returning to the target path. Below `t=1`, swapped-context rollouts are deliberately hybrid counterfactuals because their starting latent still contains the original row target.
- For the known VAE-failing condition, compare the transformer output both with Ground Truth and with the recorded VAE round-trip ceiling before interpreting two-cell errors.
- Normal 50-step generation remains the final end-to-end check; local and partial probes are diagnostics, not replacements for exact-grid generation.
